In [4]:

with open("example.txt", "w", encoding="utf-8") as file:
    file.write("Hello, World!\n")
    file.write("This is new line")

In [6]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

# 1. Запрашиваем путь у пользователя
user_input = input(
    "Введите полный путь к XML-файлу (или нажмите Enter для data.xml в текущей папке): "
).strip()

# Убираем случайные кавычки, если пользователь перетащил файл в консоль мышкой
xml_file_path = user_input.replace('"', "")
# 1. Указываем путь к файлу (учитывая вашу рабочую папку)
# Если ничего не введено, ставим путь по умолчанию
if not xml_file_path:
    xml_file_path = (
        r"C:\Users\Zelentsov Ilia\Documents\_PROGRAMMING\data.xml"
    )
# Проверим, существует ли файл, чтобы избежать ошибок
if not os.path.exists(xml_file_path):
    print(f"Ошибка: Файл не найден по пути {xml_file_path}")
else:
    # 2. Загружаем и парсим XML
    tree = ET.parse(xml_file_path)
    root = tree.getroot()

    # Сюда будем собирать данные (массив словарей)
    parsed_data = []

    # 3. Ищем все теги <item> на любом уровне вложенности (используем .iter())
    for item in root.iter("item"):
        # Извлекаем значения атрибутов. Если атрибута нет, вернется None
        name = item.get("name")
        caption = item.get("caption")
        item_type = item.get("type")  # 'type' — встроенное слово в Python, лучше назвать 'item_type'

        # Добавляем в наш массив только если нашли нужные данные
        parsed_data.append(
            {"name": name, "caption": caption, "type": item_type}
        )

    # 4. Переводим массив в таблицу Pandas для удобного анализа
    df = pd.DataFrame(parsed_data)

    # Выводим первые 5 строк в Jupyter
    print(f"Успешно спарсено элементов: {len(df)}")
    display(df.head())

    # 5. Сохраняем результат (выберите нужный вариант, убрав знак #)
    # df.to_csv('result.csv', index=False, encoding='utf-8-sig')
    # df.to_excel('result.xlsx', index=False)


Введите полный путь к XML-файлу (или нажмите Enter для data.xml в текущей папке):  devices/6.15.xml


Успешно спарсено элементов: 236


,name,caption,type
0,deviceID,Идентификатор контроллера = 6,unsigned int
1,deviceBuild,Версия прошивки контроллера = 15,unsigned int
2,deviceRev,Версия ревизии прошивки контроллера,unsigned int
3,AccessEE,Доступ на запись в EEPROM,unsigned char
4,workTime,"Время работы после включения/перзапуска, сек",unsigned long


In [7]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

# 1. Запрашиваем путь к файлу
user_input = input(
    "Введите полный путь к XML-файлу (или нажмите Enter для data.xml): "
).strip()
xml_file_path = user_input.replace('"', "")

if not xml_file_path:
    xml_file_path = (
        r"C:\Users\Zelentsov Ilia\Documents\_PROGRAMMING\data.xml"
    )

if not os.path.exists(xml_file_path):
    print(f"Ошибка: Файл не найден по пути: {xml_file_path}")
else:
    print(f"Обработка файла: {xml_file_path}")

    try:
        tree = ET.parse(xml_file_path)
        root = tree.getroot()

        parsed_rows = []

        # 2. Идем по всем тегам <device>
        for device in root.iter("device"):
            # Собираем атрибуты устройства
            device_data = {
                "device_name": device.get("name"),
                "device_caption": device.get("caption"),
                "device_id": device.get("id"),
                "device_build": device.get("build"),
            }

            # Ищем блок <ram> внутри текущего device
            ram = device.find("ram")
            if ram is None:
                continue  # Если блока ram нет, переходим к следующему устройству

            # 3. Идем по всем <item> внутри найденного <ram>
            for item in ram.findall("item"):
                # Собираем атрибуты item
                item_data = {
                    "item_name": item.get("name"),
                    "item_type": item.get("type"),
                    "item_caption": item.get("caption"),
                }

                # Проверяем наличие вложенных тегов <mode> и <EPROM> внутри item
                modes = item.findall("mode")
                eproms = item.findall("EPROM")

                # Если внутри item ничего нет, сохраняем только данные устройства и item
                if not modes and not eproms:
                    row = {**device_data, **item_data}
                    parsed_rows.append(row)
                    continue

                # 4. Если есть <mode>, создаем строки для каждого режима
                for mode in modes:
                    row = {
                        **device_data,
                        **item_data,
                        "mode_value": mode.get("value"),
                        "mode_caption": mode.get("caption"),
                        "eprom_address": None,  # Для этой строки EPROM пустой
                    }
                    parsed_rows.append(row)

                # 5. Если есть <EPROM>, создаем строки для адресов памяти
                for eprom in eproms:
                    row = {
                        **device_data,
                        **item_data,
                        "mode_value": None,
                        "mode_caption": None,
                        "eprom_address": eprom.get("address"),
                    }
                    parsed_rows.append(row)

        # 6. Создаем плоскую таблицу и сохраняем
        df = pd.DataFrame(parsed_rows)
        print(f"\nУспешно обработано строк данных: {len(df)}")

        if not df.empty:
            display(df.head(10))  # Показываем первые 10 строк

            output_csv = "parsed_devices_detailed.csv"
            df.to_csv(output_csv, index=False, encoding="utf-8-sig")
            print(f"Результат сохранен в: {os.path.abspath(output_csv)}")
        else:
            print(
                "Данные по указанной структуре <device> -> <ram> -> <item> не найдены."
            )

    except ET.ParseError:
        print(
            "Ошибка: Выбранный файл не является валидным XML (проверьте закрывающие теги)."
        )


Введите полный путь к XML-файлу (или нажмите Enter для data.xml):  devices/6.15.xml


Обработка файла: devices/6.15.xml

Успешно обработано строк данных: 68


,device_name,device_caption,device_id,device_build,item_name,item_type,item_caption,mode_value,mode_caption,eprom_address
0,IKTS11,ИКТС-11,6,15,deviceID,unsigned int,Идентификатор контроллера = 6,NaN,NaN,NaN
1,IKTS11,ИКТС-11,6,15,deviceBuild,unsigned int,Версия прошивки контроллера = 15,NaN,NaN,NaN
2,IKTS11,ИКТС-11,6,15,deviceRev,unsigned int,Версия ревизии прошивки контроллера,NaN,NaN,NaN
3,IKTS11,ИКТС-11,6,15,AccessEE,unsigned char,Доступ на запись в EEPROM,54,Доступ на запись в EEPROM открыт,NaN
4,IKTS11,ИКТС-11,6,15,workTime,unsigned long,"Время работы после включения/перзапуска, сек",NaN,NaN,NaN
5,IKTS11,ИКТС-11,6,15,RealDensity_him_sens[2],float,"Концентрация газов, измеряемая в реальном врем...",NaN,NaN,NaN
6,IKTS11,ИКТС-11,6,15,CO,float,"Фиксированная концентрация CO, ppm",NaN,NaN,NaN
7,IKTS11,ИКТС-11,6,15,NO,float,"Фиксированная концентрация NO, ppm",NaN,NaN,NaN
8,IKTS11,ИКТС-11,6,15,O2,float,"Концентрация O2 от LSU4, %об",NaN,NaN,NaN
9,IKTS11,ИКТС-11,6,15,Density_calckNOx,float,"Вычисленная концентрация NOx, ppm",NaN,NaN,NaN


Результат сохранен в: C:\Users\Zelentsov Ilia\Documents\_PROGRAMMING\Python\parsed_devices_detailed.csv
